# 🔢 Project 08 — Digit Recognizer

**What you'll build:** A neural network that can look at a tiny 8×8 pixel image of a handwritten digit and correctly identify which number (0–9) it represents.

---

## 🎯 Learning Goals
| Goal | What you'll learn |
|------|-------------------|
| 1 | How images are stored as arrays of numbers (pixels) |
| 2 | What a Multi-Layer Perceptron (MLP) neural network is |
| 3 | Why we scale/normalise features before training |
| 4 | How to evaluate a multi-class classifier (confusion matrix, accuracy) |
| 5 | How to save and reload a trained model |

⏱ **Estimated time:** 30–45 minutes  
⭐ **Difficulty:** Beginner–Intermediate


## Step 1 — Import Libraries

We need:
- **scikit-learn** for the dataset, neural network, and metrics
- **matplotlib / seaborn** for charts
- **numpy** for numerical operations
- **joblib** to save/load the trained model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
import joblib

print("All libraries imported successfully!")

## Step 2 — Load the Dataset

scikit-learn ships with a **digits dataset** — 1,797 tiny 8×8 grayscale images of handwritten digits (0–9).  
Each image is flattened to a 64-element array (64 pixel values) for machine learning.

In [ ]:
# Load the dataset
digits = load_digits()

X = digits.data      # shape: (1797, 64) — 64 pixel values per image
y = digits.target    # shape: (1797,)    — digit label 0-9
images = digits.images  # shape: (1797, 8, 8) — for visualisation

print(f"Total samples : {X.shape[0]}")
print(f"Features/image: {X.shape[1]} (8×8 pixels flattened)")
print(f"Classes       : {digits.target_names}")
print(f"\nPixel value range: {X.min():.0f} to {X.max():.0f}")

## Step 3 — Visualise Sample Digits

Each digit is only **8×8 pixels** — very low resolution, but the neural network still learns to recognise them accurately!

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(15, 5))

for digit in range(10):
    # Find first 3 occurrences of each digit
    idxs = np.where(y == digit)[0][:3]
    for row, idx in enumerate(idxs):
        axes[row, digit].imshow(images[idx], cmap='gray_r', interpolation='nearest')
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(str(digit), fontsize=12)

plt.suptitle('Sample digits (columns = digit class, rows = different examples)', y=1.02)
plt.tight_layout()
plt.savefig('sample_digits.png', dpi=100, bbox_inches='tight')
plt.show()
print("Each column is a digit class (0–9); each row is a different example.")

## Step 4 — Train / Test Split

We split the data: **80% for training**, **20% for testing**.  
`stratify=y` ensures each digit class is represented proportionally in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Keep 2-D images for the test set (used later for visualisation)
_, images_test, _, _ = train_test_split(
    images, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

## Step 5 — Feature Scaling

Neural networks train much better when input features are on similar scales.  
`StandardScaler` transforms each feature to have **mean = 0** and **std = 1**.

> ⚠️ **Important:** We fit the scaler *only* on training data, then apply it to test data. This prevents **data leakage** (accidentally letting test information influence training).

In [ ]:
scaler = StandardScaler()

# Fit on training data only
X_train_scaled = scaler.fit_transform(X_train)

# Apply (transform) to test data using the training mean/std
X_test_scaled = scaler.transform(X_test)

print(f"Before scaling — mean: {X_train.mean():.2f}, std: {X_train.std():.2f}")
print(f"After  scaling — mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")

## Step 6 — Build and Train the Neural Network

We use `MLPClassifier` — a **Multi-Layer Perceptron**, the simplest kind of neural network.

**Our network architecture:**
```
Input layer  →  64 neurons (one per pixel)
Hidden layer 1  →  128 neurons  (ReLU activation)
Hidden layer 2  →   64 neurons  (ReLU activation)
Output layer →  10 neurons (one per digit class)
```

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # Two hidden layers
    activation='relu',             # Rectified Linear Unit activation
    solver='adam',                 # Adam optimiser (good default)
    max_iter=500,                  # Up to 500 training passes
    random_state=42
)

print("Training the neural network…")
model.fit(X_train_scaled, y_train)
print(f"Training complete! Stopped after {model.n_iter_} iterations.")

## Step 7 — Evaluate the Model

We check:
- **Accuracy** — overall percentage of correct predictions
- **Classification Report** — precision, recall, F1-score per digit
- **Confusion Matrix** — which digits get confused with each other

In [ ]:
y_pred = model.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

## Step 8 — Confusion Matrix

A confusion matrix shows how many times each digit was correctly or incorrectly classified.  
- **Diagonal cells** = correct predictions (want these to be high)
- **Off-diagonal cells** = mistakes (want these to be zero or very low)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10)
)
plt.title('Confusion Matrix — Digit Recognizer', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('digit_confusion_matrix.png', dpi=100)
plt.show()

## Step 9 — Visualise Predictions

Let's see which test images the model got right (green) and wrong (red).

In [ ]:
n = 18
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
axes = axes.flatten()

for i in range(n):
    color = 'green' if y_pred[i] == y_test[i] else 'red'
    axes[i].imshow(images_test[i], cmap='gray_r', interpolation='nearest')
    axes[i].set_title(f"True:{y_test[i]} / Pred:{y_pred[i]}", color=color, fontsize=9)
    axes[i].axis('off')

plt.suptitle('Predictions (Green = Correct  |  Red = Wrong)', fontsize=12)
plt.tight_layout()
plt.savefig('digit_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

## Step 10 — Save and Reload the Model

Once trained, you can save the model to disk and reload it later — no need to retrain!

In [ ]:
# Save model and scaler
joblib.dump(model, 'digit_recognizer_model.pkl')
joblib.dump(scaler, 'digit_scaler.pkl')
print("Model and scaler saved!")

# Reload and verify
loaded_model = joblib.load('digit_recognizer_model.pkl')
loaded_scaler = joblib.load('digit_scaler.pkl')

# Quick test: predict one sample
sample = X_test[0].reshape(1, -1)
sample_scaled = loaded_scaler.transform(sample)
prediction = loaded_model.predict(sample_scaled)
print(f"\nReloaded model prediction for test sample[0]: {prediction[0]}")
print(f"True label: {y_test[0]}")

---
## 🧪 Try It Yourself!

Here are some experiments to deepen your understanding:

1. **Change the hidden layers** — try `(256,)` (one big layer) or `(64, 64, 64)` (three layers). Does accuracy change?
2. **Skip the scaler** — train without `StandardScaler`. How much does accuracy drop?
3. **Reduce training data** — use only 10% of the data (`test_size=0.9`). How does the model perform?
4. **Try a different classifier** — swap `MLPClassifier` for `RandomForestClassifier`. Compare results.
5. **Find the errors** — print all test images where `y_pred != y_test`. Can *you* tell which digit they are?

In [ ]:
# Experiment: find and display all misclassified digits
wrong_idxs = np.where(y_pred != y_test)[0]
print(f"Number of errors: {len(wrong_idxs)} out of {len(y_test)} test samples")

n_show = min(len(wrong_idxs), 12)
if n_show > 0:
    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    axes = axes.flatten()
    for i, idx in enumerate(wrong_idxs[:n_show]):
        axes[i].imshow(images_test[idx], cmap='gray_r', interpolation='nearest')
        axes[i].set_title(f"T:{y_test[idx]} P:{y_pred[idx]}", color='red', fontsize=9)
        axes[i].axis('off')
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    plt.suptitle('All Misclassified Digits', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No errors! Perfect score 🎉")

---
## 📚 Summary — What You Learned

| Concept | Plain English |
|---------|---------------|
| Pixel arrays | Images are just grids of numbers (0 = black, 16 = white for this dataset) |
| Flattening | Convert a 2-D image (8×8) into a 1-D array (64 values) for ML |
| StandardScaler | Normalise features so the network trains smoothly |
| Data leakage | Never fit the scaler on test data — only training data |
| MLPClassifier | A neural network with one or more hidden layers |
| ReLU | An activation function: output = max(0, x) — adds non-linearity |
| Adam optimiser | Smart gradient descent that adjusts learning rate automatically |
| Multi-class | 10 output neurons, one per digit; highest activation = prediction |
| Confusion matrix | Grid showing which classes get mixed up |
| joblib | Save/load trained models so you don't have to retrain every time |

**Next project →** Project 09: Weather Classifier (Random Forest + feature engineering)